In [ ]:
# ============================================================
# AI-BASED MEETING AUDIO TRANSCRIPTION & SUMMARY SYSTEM
# TEXT & SPEECH ANALYSIS PROJECT
# GOOGLE COLAB - ONE CELL COMPLETE CODE
# ============================================================


# ============================================================
# 1. INSTALL LIBRARIES
# ============================================================

!pip -q install gradio==5.44.1 faster-whisper


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import gradio as gr
import re
from collections import Counter
from faster_whisper import WhisperModel


# ============================================================
# 3. LOAD WHISPER SPEECH RECOGNITION MODEL
# ============================================================

print("=" * 60)
print("Loading Speech Recognition Model...")
print("=" * 60)

model = WhisperModel(
    "tiny",
    device="cpu",
    compute_type="int8"
)

print("✅ Whisper model loaded successfully!")
print()


# ============================================================
# 4. STOP WORDS
# ============================================================

STOP_WORDS = {
    "the", "a", "an", "is", "are", "was", "were",
    "to", "of", "and", "or", "in", "on", "for",
    "with", "this", "that", "it", "as", "at",
    "be", "by", "from", "we", "our", "you",
    "your", "i", "he", "she", "they", "their",
    "them", "have", "has", "had", "will", "would",
    "can", "could", "should", "must", "do", "does",
    "did", "but", "if", "so", "then", "than",
    "about", "into", "during", "also", "just",
    "very", "there", "here", "all", "more",
    "some", "any", "not", "no", "yes",
    "we", "our", "us", "me", "my"
}


# ============================================================
# 5. CLEAN TEXT
# ============================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ============================================================
# 6. EXTRACT KEYWORDS
# ============================================================

def extract_keywords(text, top_n=10):

    cleaned = clean_text(text)

    words = cleaned.split()

    meaningful_words = []

    for word in words:

        if word not in STOP_WORDS and len(word) > 2:
            meaningful_words.append(word)

    frequency = Counter(
        meaningful_words
    )

    return frequency.most_common(top_n)


# ============================================================
# 7. EXTRACT ACTION ITEMS
# ============================================================

def extract_action_items(text):

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    action_keywords = [
        "need to",
        "needs to",
        "should",
        "must",
        "will",
        "have to",
        "has to",
        "complete",
        "submit",
        "finish",
        "deadline",
        "prepare",
        "send",
        "review",
        "update",
        "create",
        "develop",
        "check",
        "schedule",
        "finalize"
    ]

    actions = []

    for sentence in sentences:

        sentence_clean = sentence.lower().strip()

        if len(sentence_clean) < 10:
            continue

        for keyword in action_keywords:

            if keyword in sentence_clean:

                if sentence.strip() not in actions:

                    actions.append(
                        sentence.strip()
                    )

                break

    return actions[:10]


# ============================================================
# 8. GENERATE SUMMARY
# ============================================================

def generate_summary(
    text,
    max_sentences=5
):

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text.strip()
    )

    sentences = [
        sentence.strip()
        for sentence in sentences
        if len(sentence.strip()) > 15
    ]

    if not sentences:

        return "No sufficient content available for summary."

    if len(sentences) <= max_sentences:

        return " ".join(sentences)

    keywords = extract_keywords(
        text,
        top_n=15
    )

    important_words = set()

    for word, count in keywords:

        important_words.add(word)

    scored_sentences = []

    for index, sentence in enumerate(sentences):

        words = clean_text(
            sentence
        ).split()

        score = 0

        for word in words:

            if word in important_words:

                score += 1

        if index < 3:

            score += 1

        scored_sentences.append(
            (
                score,
                index,
                sentence
            )
        )

    scored_sentences.sort(
        key=lambda x: x[0],
        reverse=True
    )

    selected = scored_sentences[
        :max_sentences
    ]

    selected.sort(
        key=lambda x: x[1]
    )

    summary_sentences = []

    for score, index, sentence in selected:

        summary_sentences.append(
            sentence
        )

    return " ".join(
        summary_sentences
    )


# ============================================================
# 9. FORMAT DURATION
# ============================================================

def format_duration(seconds):

    seconds = int(seconds)

    minutes = seconds // 60

    remaining_seconds = seconds % 60

    if minutes > 0:

        return (
            f"{minutes} min "
            f"{remaining_seconds} sec"
        )

    return f"{remaining_seconds} sec"


# ============================================================
# 10. TRANSCRIBE AUDIO
# ============================================================

def transcribe_audio(audio_path):

    segments, info = model.transcribe(
        audio_path,
        beam_size=5,
        vad_filter=True
    )

    segment_list = list(
        segments
    )

    transcript_parts = []

    for segment in segment_list:

        text = segment.text.strip()

        if text:

            transcript_parts.append(
                text
            )

    transcript = " ".join(
        transcript_parts
    ).strip()

    duration = info.duration

    return (
        transcript,
        duration
    )


# ============================================================
# 11. MAIN ANALYSIS FUNCTION
# ============================================================

def analyze_meeting(audio):

    # --------------------------------------------------------
    # Check audio
    # --------------------------------------------------------

    if audio is None:

        message = (
            "⚠️ Please upload or record "
            "a meeting audio file."
        )

        return (
            message,
            "",
            "",
            "",
            "",
            "",
            "### 🎯 Overall Score: --",
            "### 🏆 Performance Level: --"
        )


    try:

        # ----------------------------------------------------
        # Speech to Text
        # ----------------------------------------------------

        transcript, duration = (
            transcribe_audio(audio)
        )


        # ----------------------------------------------------
        # Check transcript
        # ----------------------------------------------------

        if not transcript:

            message = (
                "⚠️ No speech could be detected "
                "in the audio."
            )

            return (
                message,
                "",
                "",
                "",
                "",
                "",
                "### 🎯 Overall Score: --",
                "### 🏆 Performance Level: --"
            )


        # ----------------------------------------------------
        # Word Count
        # ----------------------------------------------------

        cleaned_transcript = clean_text(
            transcript
        )

        words = cleaned_transcript.split()

        word_count = len(words)


        # ----------------------------------------------------
        # Sentence Count
        # ----------------------------------------------------

        sentences = re.split(
            r"(?<=[.!?])\s+",
            transcript
        )

        sentences = [
            sentence.strip()
            for sentence in sentences
            if sentence.strip()
        ]

        sentence_count = len(
            sentences
        )


        # ----------------------------------------------------
        # Speaking Speed
        # ----------------------------------------------------

        minutes = duration / 60

        if minutes > 0:

            wpm = (
                word_count /
                minutes
            )

        else:

            wpm = 0


        # ----------------------------------------------------
        # Keywords
        # ----------------------------------------------------

        keyword_data = extract_keywords(
            transcript,
            top_n=10
        )

        keyword_lines = []

        for index, item in enumerate(
            keyword_data,
            start=1
        ):

            word = item[0]

            count = item[1]

            keyword_lines.append(
                f"{index}. **{word}** — "
                f"{count} occurrence(s)"
            )


        if keyword_lines:

            keyword_text = "\n".join(
                keyword_lines
            )

        else:

            keyword_text = (
                "No important keywords found."
            )


        # ----------------------------------------------------
        # Summary
        # ----------------------------------------------------

        summary = generate_summary(
            transcript,
            max_sentences=5
        )


        # ----------------------------------------------------
        # Action Items
        # ----------------------------------------------------

        actions = extract_action_items(
            transcript
        )

        action_lines = []

        for index, action in enumerate(
            actions,
            start=1
        ):

            action_lines.append(
                f"{index}. {action}"
            )


        if action_lines:

            action_text = "\n".join(
                action_lines
            )

        else:

            action_text = (
                "No possible action items detected."
            )


        # ----------------------------------------------------
        # Meeting Statistics
        # ----------------------------------------------------

        duration_text = format_duration(
            duration
        )

        keyword_count = len(
            keyword_data
        )

        action_count = len(
            actions
        )


        # ====================================================
        # SCORE CALCULATION
        # ====================================================

        # ----------------------------------------------------
        # Speaking Speed Score
        # ----------------------------------------------------

        if 100 <= wpm <= 160:

            speed_score = 100

        elif 80 <= wpm <= 180:

            speed_score = 85

        elif 60 <= wpm <= 200:

            speed_score = 70

        else:

            speed_score = 50


        # ----------------------------------------------------
        # Content Score
        # ----------------------------------------------------

        if word_count >= 500:

            content_score = 100

        elif word_count >= 250:

            content_score = 85

        elif word_count >= 100:

            content_score = 70

        else:

            content_score = 55


        # ----------------------------------------------------
        # Structure Score
        # ----------------------------------------------------

        if sentence_count >= 20:

            structure_score = 100

        elif sentence_count >= 10:

            structure_score = 85

        elif sentence_count >= 5:

            structure_score = 70

        else:

            structure_score = 55


        # ----------------------------------------------------
        # Overall Score
        # ----------------------------------------------------

        overall_score = round(
            (
                speed_score
                + content_score
                + structure_score
            ) / 3
        )


        # ----------------------------------------------------
        # Performance Level
        # ----------------------------------------------------

        if overall_score >= 85:

            level = "Excellent"

        elif overall_score >= 70:

            level = "Good"

        elif overall_score >= 50:

            level = "Average"

        else:

            level = "Needs Improvement"


        # ====================================================
        # STATISTICS MARKDOWN
        # ====================================================

        statistics = (
            "## 📊 Meeting Statistics\n\n"
            "| Metric | Result |\n"
            "|---|---:|\n"
            f"| ⏱️ Duration | **{duration_text}** |\n"
            f"| 📝 Total Words | **{word_count}** |\n"
            f"| 📄 Sentences | **{sentence_count}** |\n"
            f"| 🗣️ Speaking Speed | **{wpm:.1f} WPM** |\n"
            f"| 🔑 Important Keywords | **{keyword_count}** |\n"
            f"| ✅ Action Items | **{action_count}** |\n"
            f"| ⭐ Overall Score | **{overall_score}/100** |\n"
            f"| 🏆 Level | **{level}** |"
        )


        # ====================================================
        # SUGGESTIONS
        # ====================================================

        suggestions = []


        if wpm < 80:

            suggestions.append(
                "🗣️ Speaking speed is slow. "
                "Try to maintain a natural pace."
            )

        elif wpm > 180:

            suggestions.append(
                "🗣️ Speaking speed is high. "
                "Try speaking more slowly and clearly."
            )

        else:

            suggestions.append(
                "✅ Speaking speed is within "
                "a good range."
            )


        if word_count < 100:

            suggestions.append(
                "📝 The meeting transcript is short. "
                "Add more detailed discussion when necessary."
            )

        else:

            suggestions.append(
                "✅ The transcript contains "
                "sufficient content."
            )


        if action_count == 0:

            suggestions.append(
                "📌 No clear action items were detected. "
                "Mention tasks and deadlines explicitly."
            )

        else:

            suggestions.append(
                "✅ Action items were identified "
                "from the meeting."
            )


        suggestions_text = "\n".join(
            "- " + item
            for item in suggestions
        )


        # ====================================================
        # FINAL REPORT
        # ====================================================

        report = (
            "# 🧠 Meeting Analysis Report\n\n"
            "## ⭐ Overall Performance\n\n"
            f"**Score:** `{overall_score}/100`\n\n"
            f"**Level:** `{level}`\n\n"
            "### 💡 Suggestions\n\n"
            f"{suggestions_text}\n\n"
            "---\n\n"
            "## 🔄 TSA Processing Flow\n\n"
            "```text\n"
            "Meeting Audio\n"
            "      ↓\n"
            "Speech Recognition\n"
            "      ↓\n"
            "Speech-to-Text\n"
            "      ↓\n"
            "Text Preprocessing\n"
            "      ↓\n"
            "NLP Analysis\n"
            "      ↓\n"
            "Keyword Extraction\n"
            "      ↓\n"
            "Summary Generation\n"
            "      ↓\n"
            "Action Item Detection\n"
            "      ↓\n"
            "Meeting Report\n"
            "```"
        )


        # ====================================================
        # RETURN ALL RESULTS
        # ====================================================

        return (
            transcript,
            summary,
            keyword_text,
            action_text,
            statistics,
            report,
            f"### 🎯 Overall Score: {overall_score}/100",
            f"### 🏆 Performance Level: {level}"
        )


    except Exception as error:

        error_message = (
            "## ❌ Error\n\n"
            "Something went wrong while "
            "processing the audio.\n\n"
            "**Error Details:**\n\n"
            f"```text\n{str(error)}\n```"
        )

        return (
            error_message,
            "",
            "",
            "",
            "",
            "",
            "### 🎯 Overall Score: Error",
            "### 🏆 Performance Level: Error"
        )


# ============================================================
# 12. CUSTOM CSS
# ============================================================

css = """
.gradio-container {
    max-width: 1250px !important;
    margin: auto;
}

.main-title {
    text-align: center;
    padding: 28px;
    border-radius: 20px;
    background: linear-gradient(
        135deg,
        #5b21b6,
        #7c3aed,
        #db2777
    );
    color: white;
    margin-bottom: 20px;
}

.main-title h1 {
    font-size: 32px;
    margin-bottom: 10px;
}

.main-title p {
    font-size: 17px;
}

.info-card {
    padding: 15px;
    border-radius: 15px;
}
"""


# ============================================================
# 13. CREATE GRADIO APP
# ============================================================

with gr.Blocks(
    title="AI Meeting Audio Transcription & Summary System",
    css=css,
    theme=gr.themes.Soft()
) as app:


    # ========================================================
    # HEADER
    # ========================================================

    gr.HTML(
        """
        <div class="main-title">

            <h1>
            🧠 AI-Based Meeting Audio
            Transcription & Summary System
            </h1>

            <p>
            Text & Speech Analysis using
            Speech Recognition and Natural Language Processing
            </p>

        </div>
        """
    )


    # ========================================================
    # DESCRIPTION
    # ========================================================

    gr.Markdown(
        """
        ## 📌 About the Project

        Upload or record a meeting audio file.

        The system automatically performs:

        **Speech-to-Text → Text Processing → Keyword Extraction →
        Summary Generation → Action Item Detection → Meeting Report**
        """
    )


    # ========================================================
    # AUDIO INPUT
    # ========================================================

    audio_input = gr.Audio(
        sources=[
            "upload",
            "microphone"
        ],
        type="filepath",
        label="🎙️ Upload or Record Meeting Audio"
    )


    # ========================================================
    # ANALYZE BUTTON
    # ========================================================

    analyze_button = gr.Button(
        "🚀 Analyze Meeting",
        variant="primary",
        size="lg"
    )


    # ========================================================
    # SCORE
    # ========================================================

    with gr.Row():

        score_output = gr.Markdown(
            "### 🎯 Overall Score: --"
        )

        level_output = gr.Markdown(
            "### 🏆 Performance Level: --"
        )


    # ========================================================
    # OUTPUT TABS
    # ========================================================

    with gr.Tabs():


        # ----------------------------------------------------
        # TRANSCRIPT
        # ----------------------------------------------------

        with gr.Tab(
            "📝 Full Transcript"
        ):

            transcript_output = gr.Textbox(
                label="Meeting Transcript",
                lines=18,
                show_copy_button=True
            )


        # ----------------------------------------------------
        # SUMMARY
        # ----------------------------------------------------

        with gr.Tab(
            "📄 Summary"
        ):

            summary_output = gr.Markdown()


        # ----------------------------------------------------
        # KEYWORDS
        # ----------------------------------------------------

        with gr.Tab(
            "🔑 Keywords"
        ):

            keywords_output = gr.Markdown()


        # ----------------------------------------------------
        # ACTION ITEMS
        # ----------------------------------------------------

        with gr.Tab(
            "✅ Action Items"
        ):

            actions_output = gr.Markdown()


        # ----------------------------------------------------
        # STATISTICS
        # ----------------------------------------------------

        with gr.Tab(
            "📊 Statistics"
        ):

            statistics_output = gr.Markdown()


        # ----------------------------------------------------
        # FINAL REPORT
        # ----------------------------------------------------

        with gr.Tab(
            "📋 Final Report"
        ):

            report_output = gr.Markdown()


    # ========================================================
    # CONNECT BUTTON
    # ========================================================

    analyze_button.click(
        fn=analyze_meeting,
        inputs=audio_input,
        outputs=[
            transcript_output,
            summary_output,
            keywords_output,
            actions_output,
            statistics_output,
            report_output,
            score_output,
            level_output
        ]
    )


    # ========================================================
    # FOOTER
    # ========================================================

    gr.Markdown(
        """
        ---

        ### 🎓 Text & Speech Analysis Project

        **Technologies:**
        Python • Faster-Whisper • NLP • Gradio • Google Colab

        **Application:**
        Meeting Transcription • Summarization •
        Keyword Extraction • Action Item Detection
        """
    )


# ============================================================
# 14. LAUNCH
# ============================================================

print()
print("=" * 60)
print("🚀 Starting Gradio Application...")
print("=" * 60)
print()
print("Please wait for the public URL...")
print()

app.launch(
    share=True,
    debug=True
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.6/324.6 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocabulary.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/75.5M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

✅ Whisper model loaded successfully!


🚀 Starting Gradio Application...

Please wait for the public URL...

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ba1e40fd0dcf833335.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
